# Grab App Reviews Sentiment Analysis

## SVM + TF-IDF


### Import Library

In [1]:
!pip install sastrawi

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.7 MB/s eta 0:00:0000:01


In [2]:
import pandas as pd
import numpy as np
import csv
import requests
import json
from io import StringIO
import matplotlib.pyplot as plt
import re 
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize  
from nltk.corpus import stopwords
from wordcloud import WordCloud
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import nltk
nltk.download('punkt')  
nltk.download('stopwords') 

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

### Load Dataset

In [6]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/reviews.csv'
reviews = pd.read_csv(url)

# reviews and columns
total_reviews, total_columns  = reviews.shape
print(f"Total reviews: {total_reviews}, Total columns: {total_columns}")
print("-"*50)
reviews.info()

Total reviews: 117000, Total columns: 11
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117000 entries, 0 to 116999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   reviewId              117000 non-null  object
 1   userName              117000 non-null  object
 2   userImage             117000 non-null  object
 3   content               117000 non-null  object
 4   score                 117000 non-null  int64 
 5   thumbsUpCount         117000 non-null  int64 
 6   reviewCreatedVersion  94236 non-null   object
 7   at                    117000 non-null  object
 8   replyContent          31714 non-null   object
 9   repliedAt             31714 non-null   object
 10  appVersion            94236 non-null   object
dtypes: int64(2), object(9)
memory usage: 9.8+ MB


### Preprocessing

In [9]:
# drop nan and duplicates
reviews = reviews.dropna()
reviews = reviews.drop_duplicates()

In [10]:
# stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stem_cache = {}

# stopwords
stopword = set(stopwords.words('indonesian')) \
            | set(stopwords.words('english'))

# regex precompiled
mention = re.compile(r'@[A-Za-z0-9]+')
hashtag = re.compile(r'#[A-Za-z0-9]+')
rt = re.compile(r'\bRT\b')
url = re.compile(r'http\S+')
number = re.compile(r'\d+')
symbol = re.compile(r'[^\w\s]')

slang_url = 'https://raw.githubusercontent.com/okkyibrohim/id-abusive-language-detection/master/kamusalay.csv'
slang_df = pd.read_csv(slang_url, header=None, names=['slang', 'formal'])

slangwords = dict(zip(
    slang_df['slang'].str.lower(),
    slang_df['formal']
))

In [11]:
def cleaningText(text):
    text = mention.sub('', text)
    text = hashtag.sub('', text)
    text = rt.sub('', text)
    text = url.sub('', text)
    text = number.sub('', text)
    text = symbol.sub('', text)
    return text.strip().lower()

def stem_word(word):
    if word in stem_cache:
        return stem_cache[word]

    stemmed = stemmer.stem(word)
    stem_cache[word] = stemmed
    return stemmed

def fixSlang(text):
    # split text to words
    words = text.split()
    # slang + stopword in one pass
    words = [
        stem_word(slangwords.get(w, w))
        for w in words
        if w not in stopword
    ]
    return ' '.join(words)

In [13]:
reviews['final_text'] = reviews['content'].apply(cleaningText)
reviews['final_text'] = reviews['final_text'].apply(fixSlang)
reviews.to_csv('clean_reviews.csv', index=False)
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24020 entries, 3 to 116996
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              24020 non-null  object
 1   userName              24020 non-null  object
 2   userImage             24020 non-null  object
 3   content               24020 non-null  object
 4   score                 24020 non-null  int64 
 5   thumbsUpCount         24020 non-null  int64 
 6   reviewCreatedVersion  24020 non-null  object
 7   at                    24020 non-null  object
 8   replyContent          24020 non-null  object
 9   repliedAt             24020 non-null  object
 10  appVersion            24020 non-null  object
 11  final_text            24020 non-null  object
dtypes: int64(2), object(10)
memory usage: 2.4+ MB


### Labeling

In [14]:
def load_lexicons():
    # Positive lexicon
    positive_lexicon = {}
    pos_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv'
    response = requests.get(pos_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            positive_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch positive lexicon data")

    # Negative lexicon
    negative_lexicon = {}
    neg_url = 'https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv'
    response = requests.get(neg_url)

    if response.status_code == 200:
        reader = csv.reader(StringIO(response.text), delimiter=',')
        for row in reader:
            negative_lexicon[row[0]] = int(row[1])
    else:
        raise Exception("Failed to fetch negative lexicon data")

    return positive_lexicon, negative_lexicon

def sentiment_label(text, pos_lex, neg_lex):
    score = 0
    words = text.split()

    for w in words:
        score += pos_lex.get(w, 0)
        score += neg_lex.get(w, 0)

    return 1 if score > 0 else -1 if score < 0 else 0

In [16]:
pos_lex, neg_lex = load_lexicons()
reviews['label'] = reviews['final_text'].apply(
    lambda x: sentiment_label(x, pos_lex, neg_lex)
)
print(reviews['label'].value_counts())

label
-1    16913
 1     5798
 0     1309
Name: count, dtype: int64


In [26]:
reviews.to_csv('labeled_reviews.csv', index=False)

### Data Split and Feature Extraction

In [18]:
X = reviews['final_text']
y = reviews['label']

tfidf = TfidfVectorizer(max_features=5000, min_df=17, max_df=0.8 )
X_tfidf = tfidf.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

### Modeling

In [19]:
svm = LinearSVC(C=1.0, class_weight='balanced', random_state=42)
svm.fit(X_train.toarray(), y_train)

LinearSVC(class_weight='balanced', random_state=42)

In [20]:
y_pred_train_svm = svm.predict(X_train.toarray())
y_pred_test_svm = svm.predict(X_test.toarray())

In [21]:
accuracy_train_svm = accuracy_score(y_pred_train_svm, y_train)
accuracy_test_svm = accuracy_score(y_pred_test_svm, y_test)

In [22]:
print('SVM + TF-IDF - Train accuracy:', accuracy_train_svm)
print('SVM + TF-IDF - Test accuracy:', accuracy_test_svm)

SVM + TF-IDF - Train accuracy: 0.9485547757820864
SVM + TF-IDF - Test accuracy: 0.8974465723008604


### Testing Model / Inference

In [23]:
texts = [
    "Driver gofood sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [24]:
X_new = tfidf.transform(texts)
y_pred_new = svm.predict(X_new.toarray())

label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, label in zip(texts, y_pred_new):
    print("Text:", text)
    print("Prediction:", label_map[label])
    print("-" * 50)

Text: Driver gofood sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


In [25]:
import joblib
joblib.dump(svm, '/kaggle/working/svm_sentiment_model.pkl')

['/kaggle/working/svm_sentiment_model.pkl']

## BiLSTM + Word2Vec

### Import Libraries

In [42]:
!pip install -q gensim

In [97]:
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Load Dataset

In [46]:
url = 'https://raw.githubusercontent.com/bluga404/sa-grab/main/labeled_reviews.csv'
reviews = pd.read_csv(url)
reviews = reviews.dropna()
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24018 entries, 0 to 24019
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              24018 non-null  object
 1   userName              24018 non-null  object
 2   userImage             24018 non-null  object
 3   content               24018 non-null  object
 4   score                 24018 non-null  int64 
 5   thumbsUpCount         24018 non-null  int64 
 6   reviewCreatedVersion  24018 non-null  object
 7   at                    24018 non-null  object
 8   replyContent          24018 non-null  object
 9   repliedAt             24018 non-null  object
 10  appVersion            24018 non-null  object
 11  final_text            24018 non-null  object
 12  label                 24018 non-null  int64 
dtypes: int64(3), object(10)
memory usage: 2.6+ MB


### Data Split and Feature Extraction

In [48]:
# get text
texts = reviews['final_text']
# convert text to list
texts = texts.tolist()
# apply tokenization for every sentence in text list
tokenized_text = [word_tokenize(sentence) for sentence in texts]

In [50]:
w2v_model = Word2Vec(
    window = 10,
    min_count = 5,
    workers = 4,
    epochs = 10
)

w2v_model.build_vocab(tokenized_text, progress_per=1000)
w2v_model.train(tokenized_text, total_examples=w2v_model.corpus_count, epochs=w2v_model.epochs)
w2v_model.save('word2vec-indo.model')

In [53]:
w2v_model = Word2Vec.load('word2vec-indo.model')
w2v_model.wv.most_similar("lambat")

[('undur', 0.771282434463501),
 ('proses', 0.7218286991119385),
 ('tunggu', 0.7208125591278076),
 ('datang', 0.7102159857749939),
 ('wita', 0.7009022831916809),
 ('jadwal', 0.700032114982605),
 ('estimasi', 0.6899662613868713),
 ('perut', 0.6796556711196899),
 ('mundur', 0.672099232673645),
 ('day', 0.6610552668571472)]

In [54]:
print("vocab length:", len(w2v_model.wv.key_to_index))

vocab length: 3404


In [57]:
length = [len(tokens) for tokens in tokenized_text]
print(f"longest: {max(length)} words")
print(f"shortest: {min(length)} words")
print(f"shortest: {np.median(length)} words")

longest: 100 words
shortest: 0 words
shortest: 12.0 words


In [98]:
vocab_size = 10000
max_len = 50 
embedding_dim = w2v_model.vector_size

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<oov>")
tokenizer.fit_on_texts(tokenized_text)
word_index = tokenizer.word_index
print(f"there are {len(word_index)} unique tokens")

there are 22860 unique tokens


In [99]:
sequences = tokenizer.texts_to_sequences(tokenized_text)
X_data = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

In [100]:
# +1 cause index 0 reserved for padding
num_words = min(vocab_size, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))

found_words = 0
for word, i in word_index.items():
    if i > vocab_size:
        continue

    if word in w2v_model.wv:
        embedding_matrix[i] = w2v_model.wv[word]
        found_words += 1
    else:
        pass

print(f"mapped {found_words} from word2vec to keras")

mapped 3404 from word2vec to keras


In [101]:
y = reviews['label'].values
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y,
    test_size = 0.3,
    stratify=y
)

# shift labels to be 0, 1, 2
y_train = y_train + 1
y_test = y_test + 1

In [102]:
print(f"training data shape: {X_train.shape}")
print(f"test data shape: {X_test.shape}")

training data shape: (16812, 50)
test data shape: (7206, 50)


### Modeling

In [103]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=num_words,
        output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        input_length=max_len,
        trainable=True
    )
)

# Bi-LSTM Layer
model.add(Bidirectional(LSTM(64, return_sequences=True, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

model.add(Bidirectional(LSTM(32, return_sequences=False, kernel_regularizer=l2(0.001))))
model.add(Dropout(0.5))

# Dense Layer
model.add(Dense(32, activation='relu'))

# Output Layer
model.add(Dense(3, activation='softmax'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_10                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_11                │ ?                      │   0 (unbuilt) │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [104]:
optimizer = Adam(learning_rate=1e-4)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',      
    patience=3,              
    restore_best_weights=True
)

In [105]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,     
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.6643 - loss: 1.3201 - val_accuracy: 0.7788 - val_loss: 0.9552
Epoch 2/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.7720 - loss: 0.9475 - val_accuracy: 0.8139 - val_loss: 0.7778
Epoch 3/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8083 - loss: 0.7873 - val_accuracy: 0.8282 - val_loss: 0.6802
Epoch 4/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8349 - loss: 0.6747 - val_accuracy: 0.8549 - val_loss: 0.6001
Epoch 5/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8628 - loss: 0.5752 - val_accuracy: 0.8686 - val_loss: 0.5349
Epoch 6/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8853 - loss: 0.5023 - val_accuracy: 0.8716 - val_loss: 0.5038
Epoch 7/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.8968 - loss: 0.4502 - val_accuracy: 0.8793 - val_loss: 0.4719
Epoch 8/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9140 - loss: 0.3900 - val_acc

In [106]:
results = model.evaluate(X_test, y_test, batch_size=32)
print(f"Test Loss: {results[0]:.4f}")
print(f"Test Accuracy: {results[1]*100:.2f}%")

226/226 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9023 - loss: 0.4037
Test Loss: 0.4050
Test Accuracy: 90.23%


### Testing Model / Inference

In [107]:
texts = [
    "Driver gofood sangat buruk dan mengecewakan",
    "pesanan saya dibatalkan sepihak tanpa alasan yg jelas",
    "Promo mudah digunakan dan proses pemesanan lancar",
    "saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya",
    "Pelayanan sangat memuaskan, terima kasih grab"
]

In [111]:
processed_texts = []

for text in texts:
    text = str(text).lower()
    fixed = fixSlang(text)
    processed_texts.append(fixed)

final_tokens = [word_tokenize(t) for t in processed_texts]
seqs = tokenizer.texts_to_sequences(final_tokens)
X_new = pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

predictions = model.predict(X_new)
y_pred_indices = np.argmax(predictions, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step


In [112]:
label_map = {
    -1: "Negative",
     0: "Netral",
     1: "Positive"
}

for text, index in zip(texts, y_pred_indices):
    # Convert 0,1,2 back to -1,0,1
    original_label = index - 1 
    sentiment = label_map[original_label]
    
    print(f"Text: {text}")
    print(f"Prediction: {sentiment}")
    print("-" * 50)

Text: Driver gofood sangat buruk dan mengecewakan
Prediction: Negative
--------------------------------------------------
Text: pesanan saya dibatalkan sepihak tanpa alasan yg jelas
Prediction: Negative
--------------------------------------------------
Text: Promo mudah digunakan dan proses pemesanan lancar
Prediction: Positive
--------------------------------------------------
Text: saya sudah pesan gofood, tapi selama 2 jam tdak sampai juga, drivernya marah lagi pas ketemu saya
Prediction: Negative
--------------------------------------------------
Text: Pelayanan sangat memuaskan, terima kasih grab
Prediction: Positive
--------------------------------------------------


## IndoBERT-Base